In [ ]:
import os
import time
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Cargar la API Key desde el archivo .env local
load_dotenv()

# ==========================================
# 1. CONFIGURACIÓN DE LA API Y CONFIGS
# ==========================================
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("Error: No se encontró la 'GEMINI_API_KEY' en el archivo .env.")
    print("Por favor, asegúrate de crear el archivo .env en la raíz del proyecto y colocar tu llave.")
    exit()

try:
    client = genai.Client(api_key=api_key)
except Exception as e:
    print(f"Error al conectar con el cliente de Gemini: {e}")
    exit()

# Variables de archivos sincronizadas con tu script real
ARCHIVO_ENTRADA = "datos_keto.csv"
ARCHIVO_SALIDA = "datos_evaluados_keto.csv"
TAMANO_MUESTRA = 30

# ==========================================
# 2. CARGA DE DATOS Y MUESTREO (MAS)
# ==========================================
if not os.path.exists(ARCHIVO_ENTRADA):
    print(f"Error: No se encontró el archivo '{ARCHIVO_ENTRADA}'.")
    print("Por favor, ejecuta primero tu script de scraping para consolidar los datos.")
    exit()

print(f"Cargando datos desde {ARCHIVO_ENTRADA}...")
df_completo = pd.read_csv(ARCHIVO_ENTRADA)

# Aplicamos Muestreo Aleatorio Simple (MAS)
print(f"Seleccionando una muestra aleatoria de {TAMANO_MUESTRA} registros...")
# Nota: Si la base total tiene menos de 30 registros, se evalúa el total disponible sin romper el script
n_muestras = min(TAMANO_MUESTRA, len(df_completo))
df_muestra = df_completo.sample(n=n_muestras, random_state=42).copy()

# ==========================================
# 3. EL MOTOR DE IA (PROMPT ESTRICTO)
# ==========================================
PROMPT_SISTEMA = (
    "Eres un clasificador estricto de desinformación médica y de nutrición. "
    "Tu tarea es evaluar el texto que te proporcione el usuario.\n\n"
    "Reglas de clasificación:\n"
    "- Responde 'ALERTA' si el texto contiene desinformación, mitos, teorías de conspiración o promesas milagrosas sobre la salud/dieta sin sustento real.\n"
    "- Responde 'CONFIABLE' si el texto es informativo, mantiene neutralidad o cita/basa su información en fuentes médicas o científicas contrastadas.\n\n"
    "CRÍTICO: Responde ÚNICAMENTE con la palabra 'ALERTA' o 'CONFIABLE'. No agregues introducciones, explicaciones, signos de puntuación ni texto adicional."
)

evaluaciones = []

print("\nIniciando evaluación automatizada con Gemini...")
for index, fila in df_muestra.iterrows():
    # Mapeo directo con las columnas exactas de tu CSV
    texto_articulo = fila.get('Texto', '')
    titulo_articulo = fila.get('Título', 'Sin título')
    
    if pd.isna(texto_articulo) or str(texto_articulo).strip() == "":
        print(f"Fila {index}: Texto vacío. Saltando...")
        evaluaciones.append("NO_EVALUABLE")
        continue
        
    print(f"Evaluando: '{str(titulo_articulo)[:40]}...' ", end="", flush=True)
    
    try:
        # Llamada optimizada usando gemini-2.5-flash
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=f"Texto a evaluar:\n{texto_articulo}",
            config=types.GenerateContentConfig(
                system_instruction=PROMPT_SISTEMA,
                temperature=0.0  # Consistencia y rigidez en la respuesta de clasificación
            )
        )
        
        resultado = response.text.strip().upper()
        print(f"-> [Resultado: {resultado}]")
        evaluaciones.append(resultado)
        
    except Exception as e:
        print(f"-> [Error en API: {e}]")
        evaluaciones.append("ERROR_API")
    
    # Control de flujo para la cuota gratuita de la API (2 segundos de espera)
    time.sleep(2)

# ==========================================
# 4. GUARDAR RESULTADOS
# ==========================================
df_muestra['Evaluacion_IA'] = evaluaciones
df_muestra.to_csv(ARCHIVO_SALIDA, index=False, encoding='utf-8')

print(f"\nFase 3 Completada con éxito.")
print(f"Resultados guardados localmente en el archivo: '{ARCHIVO_SALIDA}'")